# Dự báo Doanh thu Sản phẩm
## 4. Phân tích Khám phá Dữ liệu (EDA)

Chúng ta tiến hành phân tích trực quan hóa để hiểu sâu hơn về phân phối của doanh thu, đặc điểm nhân khẩu học của khách hàng, và hành vi mua sắm tại các trung tâm thương mại theo thời gian.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

CLEANED_DATA_PATH = r'../data/preprocessed-data/preprocessed_customer_shopping_data.csv'
df = pd.read_csv(CLEANED_DATA_PATH, parse_dates=['invoice_date'])
print('Kích thước bộ dữ liệu:', df.shape)
df.head()

### 4.1 Phân phối biến mục tiêu Doanh thu (Sales_Revenue)
Chúng ta phân tích tần suất phân phối doanh thu trên mỗi giao dịch và xem xét độ lệch.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Sales_Revenue'], bins=50, kde=True, color='teal')
plt.title('Phân phối của Doanh thu giao dịch', fontsize=14, fontweight='bold')
plt.xlabel('Doanh thu (Liras)')
plt.ylabel('Tần suất')
plt.tight_layout()
plt.show()

print(f'Độ lệch (Skewness) của doanh thu: {df["Sales_Revenue"].skew():.2f}')

### 4.2 Phân phối độ tuổi và giới tính khách hàng

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['age'], bins=20, kde=True, color='skyblue', ax=axes[0])
axes[0].set_title('Phân phối Độ tuổi khách hàng', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tuổi')

gender_counts = df['gender'].value_counts()
axes[1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['lightcoral', 'skyblue'], startangle=90, wedgeprops=dict(edgecolor='black'))
axes[1].set_title('Phân phối Giới tính', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 4.3 Thống kê số lượng giao dịch theo danh mục, thanh toán và trung tâm thương mại

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 15))

sns.countplot(data=df, y='category', order=df['category'].value_counts().index, palette='viridis', ax=axes[0])
axes[0].set_title('Số lượng giao dịch theo Danh mục sản phẩm', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Số lượng hóa đơn')

sns.countplot(data=df, x='payment_method', order=df['payment_method'].value_counts().index, palette='Set2', ax=axes[1])
axes[1].set_title('Số lượng giao dịch theo Phương thức thanh toán', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Phương thức thanh toán')

sns.countplot(data=df, y='shopping_mall', order=df['shopping_mall'].value_counts().index, palette='coolwarm', ax=axes[2])
axes[2].set_title('Số lượng giao dịch theo Trung tâm thương mại (Mall)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Số lượng hóa đơn')

plt.tight_layout()
plt.show()

### 4.4 Phân phối doanh thu theo từng trung tâm thương mại

In [ ]:
plt.figure(figsize=(14, 8))
sns.boxplot(data=df, x='Sales_Revenue', y='shopping_mall', palette='Set3')
plt.title('Phân phối Doanh thu giao dịch trên từng Trung tâm thương mại', fontsize=14, fontweight='bold')
plt.xlabel('Doanh thu (Liras)')
plt.ylabel('Trung tâm thương mại')
plt.tight_layout()
plt.show()

### 4.5 Phân tích hành vi mua sắm theo thời gian
Chúng ta phân tích tần suất giao dịch thay đổi như thế nào giữa các tháng và các ngày trong tuần.

In [ ]:
df['YearMonth'] = df['invoice_date'].dt.to_period('M')
df['DayOfWeek'] = df['invoice_date'].dt.day_name()

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

monthly_tx = df.groupby('YearMonth').size()
monthly_tx.plot(kind='line', marker='o', color='teal', ax=axes[0])
axes[0].set_title('Xu hướng số lượng giao dịch hàng tháng (T1/2021 - T2/2023)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Số lượng hóa đơn')

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.countplot(data=df, x='DayOfWeek', order=day_order, palette='magma', ax=axes[1])
axes[1].set_title('Số lượng giao dịch theo ngày trong tuần', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Ngày trong tuần')

plt.tight_layout()
plt.show()

### 4.6 Xu hướng giao dịch tuần tự tại các trung tâm thương mại

In [ ]:
plt.figure(figsize=(14, 6))
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_mall = df.groupby(['DayOfWeek', 'shopping_mall']).size().reset_index(name='Transaction_Count')
sns.lineplot(data=daily_mall, x='DayOfWeek', y='Transaction_Count', hue='shopping_mall', marker='o', sort=False)
plt.title('Xu hướng hóa đơn các ngày trong tuần theo từng Mall', fontsize=14, fontweight='bold')
plt.ylabel('Số lượng hóa đơn')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 4.7 Các kết luận cốt lõi rút ra từ EDA
1. **Nhu cầu thị trường ổn định:** Khối lượng giao dịch duy trì ở mức biên độ hẹp cực kỳ phẳng qua các tháng (khoảng 3.500 đến 4.000 hóa đơn mỗi tháng trên mỗi trung tâm thương mại) và không có đột biến vào cuối năm.
2. **Hành vi mua sắm đồng đều:** Số lượng khách hàng mua sắm không có sự chênh lệch lớn giữa các ngày trong tuần và ngày cuối tuần.
3. **Biến doanh thu lệch phải:** `Sales_Revenue` có độ lệch rất cao (độ lệch 2.87) với nhiều đỉnh nhỏ tương ứng với các tổ hợp đơn giá và số lượng rời rạc.

Chuyển sang **Bước 5: Feature Engineering**.